# Шаг 3: Сборка `scorer.jsonl` файлов

Для каждой из 52 интрукции собираем предпочтения наших ощенщиков. По карточке проекта их двя -- gpt-4 и claude, поэтому по в файле содержатся две строки на каждую инструкцию. 

```json
{"id":"ALP-0001","scorer":"gpt4","verdict":"A","preference_raw":1}
{"id":"ALP-0001","scorer":"claude","verdict":"A","preference_raw":1}
```

Поле `verdict` определяем следующим образом:
- `A` если `preference = 1`, 
- `B` если `preference = 2`, 
- `TIE` иначе.

In [7]:
import pandas as pd
import os
import json

slice = pd.read_json('audit/slice.jsonl', lines = True)
comparable = pd.read_json('full_dataset.json')

def pref_to_verdict(p):
    if p == 1: return "A"
    if p == 2: return "B"
    return "TIE"

# slice_ids + предпочтения из comparable по тем же ключам
slice_df = pd.DataFrame(slice)
key_cols = ["instruction","output_1","output_2","generator_1","generator_2"]
prefs = comparable.set_index(key_cols)[["pref_gpt4","pref_claude"]]
slice_df = slice_df.merge(prefs, left_on=key_cols, right_index=True, how="left")

with open("audit/scorer.jsonl", "w", encoding="utf-8") as f:
    for _, r in slice_df.iterrows():
        f.write(json.dumps({"id": r["id"], "scorer":"gpt4", "verdict": pref_to_verdict(r["pref_gpt4"]),
                            "preference_raw": int(r["pref_gpt4"])}, ensure_ascii=False) + "\n")
        f.write(json.dumps({"id": r["id"], "scorer":"claude", "verdict": pref_to_verdict(r["pref_claude"]),
                            "preference_raw": int(r["pref_claude"])}, ensure_ascii=False) + "\n")